# Full-run test for ADASYN

This notebook provides runnable cells to perform a full training run for 'ADASYN' using the project `TrainTestSplitPipeline`.

Notes:
- ADASYN is adaptive and generates more samples for hard-to-learn minority examples
- Similar to SMOTE but with density-based weighting

In [1]:
# Install dependencies
%pip install imbalanced-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [3]:
# Configuration
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']
SKIP_EVALUATIONS = False  # Set to True to skip TSTR evaluations

# Top-level dirs
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

MODEL_MAP = {
    'adasyn': ('katabatic.models.adasyn.models', 'ADASYNModel'),
}

ADASYN_CONFIG = {
    'n_neighbors': 5,
    'sampling_strategy': 'auto',
    'random_state': 42,
}

## Preprocess datasets (run once)

In [4]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(
            file_path=f'raw_data/{dataset}.csv', 
            output_path=f'discretized_data/{dataset}.csv', 
            bins=10, 
            strategy='uniform'
        )
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        traceback.print_exc()


Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
Discretized -> discretized_data/shuttle.csv


## Run ADASYN

In [5]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'ADASYN -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'adasyn')
    ensure(synth_dir)
    try:
        mod_path, cls_name = MODEL_MAP['adasyn']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)
        model_factory = lambda: ModelClass(**ADASYN_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        print('ADASYN finished for', dataset)
    except Exception as e:
        print('ADASYN failed for', dataset, e)
        traceback.print_exc()


ADASYN -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[ADASYN] Initializing with n_neighbors=5...
[ADASYN] Ready to generate samples from 26048 training samples...
[ADASYN] Generated samples in 2.34 seconds.
[ADASYN] Generated 13520 new synthetic samples...
[ADASYN] Returning 26048 total samples (original size with balanced classes)...
[ADASYN] Synthetic data saved:
  X -> synthetic\adult\adasyn\x_synth.csv
  y -> synthetic\adult\adasyn\y_synth.csv

Results saved to: Results\adult\adasyn_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7141
F1 Score: 0.7343
AUC: 0.8322

MLP:
Accuracy: 0.7712
F1 Score: 0.7861
AUC: 0.8778

RF:
Accuracy: 0.7921
F1 